# Visualization of Model Performance

In [31]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.animation as animation
import numpy as np
import os
from glob import glob
import torch
from model import GNN_mtl_gnn
from torch_geometric.data import Data

In [34]:
def rotation_matrix_back(yaw):
    """
    Rotate back. 
    https://en.wikipedia.org/wiki/Rotation_matrix#Non-standard_orientation_of_the_coordinate_system
    """
    rotation = np.array([[np.cos(-np.pi/2+yaw), -np.sin(-np.pi/2+yaw)],[np.sin(-np.pi/2+yaw), np.cos(-np.pi/2+yaw)]])
    rotation = torch.tensor(rotation).float()
    return rotation

def build_data_for_track(track_rows, obs_len, device):
    """
    track_rows: deque with dict-like rows (last obs_len already kept in order).
    Creates Data.x  [obs_len, 7]  =  [X, Y, speed, yaw, int_left, int_right, int_straight]
    """
    # build intent flags from the first stored row
    tid = track_rows[0]['TRACK_ID']
    if 'left' in tid:
        intent = [1, 0, 0]
    elif 'right' in tid:
        intent = [0, 1, 0]
    else:
        intent = [0, 0, 1]

    feats = [[r['X'], r['Y'], r['speed'], r['yaw'], *intent] for r in track_rows]
    x = torch.tensor(feats, dtype=torch.float, device=device)
    edge_index = torch.empty((2, 0), dtype=torch.long, device=device)     # no graph edges here
    data = Data(x=x, edge_index=edge_index)
    print(f"Data contains {data.x.shape[0]} frames for track {tid} with features: {data.x.shape[1]}")
    return data

In [35]:
# --- data and weights paths ---
csv_folder = 'csv/train_1k_simple_separate_10m'
model_path = 'trained_params_archive/sumo_with_mpc_online_control/model_rot_gnn_mtl_np_sumo_0911_e3_1930.pth'

# --- load your trained model ---
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = GNN_mtl_gnn(hidden_channels=128).to(device)
model.load_state_dict(torch.load(model_path, map_location=device))
model.eval()

all_csvs = sorted(glob(os.path.join(csv_folder, "*.csv")))

if not all_csvs:
    print("No CSV files found.")
    raise FileNotFoundError(f"No CSV files found in {csv_folder}")

df = pd.concat([pd.read_csv(csv_path) for csv_path in all_csvs])
df = df.sort_values(by='TIMESTAMP')

# Normalize track IDs to assign each car a unique color
track_ids = df['TRACK_ID'].unique()
colors = {track_id: plt.cm.tab20(i % 20) for i, track_id in enumerate(track_ids)}

In [37]:
df.head()

,Unnamed: 0,TIMESTAMP,TRACK_ID,OBJECT_TYPE,X,Y,yaw,speed,CITY_NAME
0,167,7.4,down_up_1,tgt,5.0,-53.36,0.0,12.68,SUMO
1,171,7.5,down_up_1,tgt,5.0,-52.07,0.0,12.88,SUMO
2,175,7.6,down_up_1,tgt,5.0,-50.77,0.0,12.98,SUMO
3,179,7.7,down_up_1,tgt,5.0,-49.48,0.0,12.98,SUMO
4,183,7.8,down_up_1,tgt,5.0,-48.18,0.0,12.99,SUMO


In [10]:
# Group by timestamp
timestamps = sorted(df['TIMESTAMP'].unique())
timestamps[:15]

[7.4, 7.5, 7.6, 7.7, 7.8, 7.9, 8.0, 8.1, 8.2, 8.3, 8.4, 8.5, 8.6, 8.7, 8.8]

In [38]:
grouped = df.groupby('TIMESTAMP')


In [ ]:
# Setup plot
fig, ax = plt.subplots(figsize=(8, 8))
ax.set_xlim(df['X'].min() - 10, df['X'].max() + 10)
ax.set_ylim(df['Y'].min() - 10, df['Y'].max() + 10)
ax.set_aspect('equal')

# Set of track currently in the scene:
shown_tracks = set()

for frame in range(len(timestamps)):
    ts = timestamps[frame]
    ax.clear()
    ax.set_xlim(df['X'].min() - 10, df['X'].max() + 10)
    ax.set_ylim(df['Y'].min() - 10, df['Y'].max() + 10)
    ax.set_title(f"Time: {ts:.2f} seconds")
    ax.set_xlabel('X')
    ax.set_ylabel('Y')
    ax.set_aspect('equal') # Maintain aspect ratio
    
    frame_df = grouped.get_group(ts)

    for _, row in frame_df.iterrows():
        track_id = row['TRACK_ID']
        
        # If the track is new in the current frame, take the data from the dataframe
        if track_id not in shown_tracks:
            shown_tracks.add(track_id)
            color = colors[track_id]
            ax.plot(row['X'], row['Y'], 'o', color=color, label=f'Track {track_id}')

        # If the track is already shown, just update its position using the model's prediction
        else:
            with torch.no_grad():
                data = build_data_for_track([row], obs_len=30, device=device)
                out = model(data.x[:, [0, 1, 4, 5, 6]], data.edge_index)

    break
        # if track_id not in shown_tracks:


Unnamed: 0           167
TIMESTAMP            7.4
TRACK_ID       down_up_1
OBJECT_TYPE          tgt
X                    5.0
Y                 -53.36
yaw                  0.0
speed              12.68
CITY_NAME           SUMO
Name: 0, dtype: object
